In [1]:
%load_ext autoreload
%autoreload 2

## Collecting verb transactions NER and TIMEX data

As a result of running the script, a database file `timex_ner.db`, is generated.

Input:

- Verb transactions database: `v33.db`.

Output:

- Verb transactions Timex and NER database: `timex_ner.db`.

### Table 1

**timex**

| Väli             | Tüüp | Kirjeldus                 | Näide | Märkus |
| ---------------- | ---- | ------------------------- | ----- | ------ |
| id               | int  | rea unikaalne ID          |       |        |
| sentence_id      | int  | lause id korpuses         |       |        |
| loc              | int  | sõna positsioon lauses    |       |        |
| timex_id         | int  | TIMEX fraasi id lauses    |       |        |
| timex_type       | str  | TIMEX tüüp                |       |        |
| part_of_interval | str  | TIMEX tüüp                |       |        |
| timex_members    | int  | TIMEX fraasi liikmete arv |       |        |

### Table 2

**ner**

| Väli        | Tüüp | Kirjeldus               | Näide | Märkus |
| ----------- | ---- | ----------------------- | ----- | ------ |
| id          | int  | rea unikaalne ID        |       |        |
| sentence_id | int  | lause id korpuses       |       |        |
| loc         | int  | sõna positsioon lauses  |       |        |
| ner_id      | int  | NER fraasi id lauses    |       |        |
| ner_tag     | int  | NER taglauses           |       |        |
| ner_members | int  | NER fraasi liikmete arv |       |        |


In [2]:
import os
import sys
import pandas as pd
import math
import helpers.collect as collect

from sqlalchemy import create_engine, text
from sqlalchemy.orm import  Session
from estnltk.storage.postgres import PostgresStorage, IndexQuery
from helpers.syntax_graph import SyntaxGraph
from helpers.models import Base, Ner, Timex
from datetime import datetime
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent.parent
sys.path.append(ROOT / "common_code")



## Input parameters

In [ ]:

# verb transactions database
#TRANSACTION_DB = ROOT / "databases/v33_data.db"
TRANSACTION_DB = Path("./example_data/transactions.db")

# result database with ner and timex data
RESULT_DB = "./example_data/ner_timex.db"
#date_time = datetime.now().strftime("%Y%m%d-%H%M%S")
#RESULT_DB = f"ner_timex_{date_time}.db"

# layers used for data extracted
LAYERS = ["v171_named_entities", "v172_stanza_syntax", "v172_pre_timexes"]

# collection of koondkorpus sentences
COLLECTION_NAME = 'koondkorpus_sentences'

# batch size to write in local db-file
BATCH_SIZE = 1000

# batch size of reading vert trx ids from verb transactions database
BATCH_SIZE_TRX_ROW = 100000


## Connect to db

In [4]:
# create db
engine = create_engine(f"sqlite+pysqlite:///{RESULT_DB}", future=True, echo=False)
conn = engine.connect()


# connect koondkorpus sentences db
storage = PostgresStorage(
    pgpass_file='~/.pgpass', schema="estonian_text_corpora", role="estonian_text_corpora_read", temporary=False
)



2025-10-21 11:11:58 INFO     connecting to host: 'postgres.keeleressursid.ee', port: '5432', dbname: 'estonian-text-corpora', user: 'zummy'
2025-10-21 11:11:58 INFO     schema: 'estonian_text_corpora', temporary: False, role: 'estonian_text_corpora_read'


INFO:storage.py:57: connecting to host: 'postgres.keeleressursid.ee', port: '5432', dbname: 'estonian-text-corpora', user: 'zummy'
INFO:storage.py:108: schema: 'estonian_text_corpora', temporary: False, role: 'estonian_text_corpora_read'


## Creating tables

In [5]:


# tables are defined in models.py
with Session(bind=conn) as session:
    Base.metadata.create_all(engine)
    session.commit()
    print("Database and tables created.")

Database and tables created.


## Workflow

- Read batch of sentence_ids + verb phrase members locs from `transactions.transaction_head` and `transactions.transaction_row` tables.
- Fetch setences layers from koondkorpus database.
- Extract TIMEX and NER info for verbphrase nodes.
- Store info to local db.

In [6]:
collection = storage[COLLECTION_NAME]
logger = collect.logger

with Session(engine) as session:
    # attaching trnx database
    db_path = os.fspath(TRANSACTION_DB)  # or str(TRANSACTION_DB)
    session.execute(text("ATTACH DATABASE :trxdb AS transactions"), {"trxdb": db_path})
    
    total_trnx_rows = collect.get_total_rows_to_fetch(session)
    
    batches_total =  math.ceil(total_trnx_rows/BATCH_SIZE_TRX_ROW)

    sentences_checked = 0

    logger.info(f"Transaction rows total: {total_trnx_rows}")
    logger.info(f"Batches to fetch: {batches_total}")


    collected_ner = []
    collected_timex = []

    # last trx_row_id
    for batch_nr in range(1, batches_total+1):
        offset = (batch_nr-1)*BATCH_SIZE_TRX_ROW
        logger.info(f"BATCH: {batch_nr}. Limit {BATCH_SIZE_TRX_ROW}, offset {offset}")
        
        sentence_ids = collect.extract_sentence_and_nodes_verbs(session, batch_size=BATCH_SIZE_TRX_ROW, offset=offset)
        my_sentence_ids = list(sentence_ids.keys())
        logger.info(f"Sentences to fetch: {len(my_sentence_ids)}")

        # request data of relevant sentences
        my_select = collection.select(
            IndexQuery(my_sentence_ids), progressbar=None, layers=LAYERS, return_index=True
        )

        setneces = 0
        for col_id, text_data in my_select:
            sentences_checked += 1

            # whitelisted nodes
            wl_nodes = sentence_ids[col_id]
            
            timex, ner = collect.collect_data(col_id=col_id, text=text_data, nodes=wl_nodes)

            collected_timex = collected_timex + timex
            collected_ner = collected_ner + ner

        if len(collected_ner) > BATCH_SIZE:
            collect.save_ner_to_db(session, collected_ner)
            collected_ner = []
            
        if len(collected_timex) > BATCH_SIZE:
            collect.save_timex_to_db(session, collected_timex)
            logger.info("write timex to db")
            collected_timex = []


    df_ner = pd.DataFrame(collected_ner)
    df_timex = pd.DataFrame(collected_timex)
  
    if len(collected_ner):
        collect.save_ner_to_db(session, collected_ner)
            
       
    if len(collected_timex):
        collect.save_timex_to_db(session, collected_timex)
        
        
    logger.info(f"Sentences checked: {sentences_checked}")
    logger.info("Done.")


2025-10-21 11:11:59 INFO     Transaction rows total: 837
2025-10-21 11:11:59 INFO     Batches to fetch: 1
2025-10-21 11:11:59 INFO     BATCH: 1. Limit 100000, offset 0
2025-10-21 11:11:59 INFO     Starting fetching sentence and node ids for batch.
2025-10-21 11:11:59 INFO     Fetched 837 rows, unique sentence ids: 286
	First row: {'head_id': 3, 'trnx_row_id': 4, 'sentence_id': 5, 'verb_position': 11, 'child_position': 1}
	Last row: {'head_id': 30046800, 'trnx_row_id': 54001020, 'sentence_id': 21360895, 'verb_position': 4, 'child_position': 3}
2025-10-21 11:11:59 INFO     Sentences to fetch: 286


INFO:1384940255.py:15: Transaction rows total: 837
INFO:1384940255.py:16: Batches to fetch: 1
INFO:1384940255.py:25: BATCH: 1. Limit 100000, offset 0
INFO:collect.py:155: Starting fetching sentence and node ids for batch.
INFO:collect.py:192: Fetched 837 rows, unique sentence ids: 286
	First row: {'head_id': 3, 'trnx_row_id': 4, 'sentence_id': 5, 'verb_position': 11, 'child_position': 1}
	Last row: {'head_id': 30046800, 'trnx_row_id': 54001020, 'sentence_id': 21360895, 'verb_position': 4, 'child_position': 3}
INFO:1384940255.py:29: Sentences to fetch: 286


2025-10-21 11:12:03 INFO     NER, saving to db
2025-10-21 11:12:03 INFO     NER, saved to db 32 rows
2025-10-21 11:12:03 INFO     Timex, saving to db
2025-10-21 11:12:03 INFO     Timex, saved to db 22 rows
2025-10-21 11:12:03 INFO     Sentences checked: 269
2025-10-21 11:12:03 INFO     Done.


INFO:collect.py:220: NER, saving to db
INFO:collect.py:223: NER, saved to db 32 rows
INFO:collect.py:213: Timex, saving to db
INFO:collect.py:216: Timex, saved to db 22 rows
INFO:1384940255.py:69: Sentences checked: 269
INFO:1384940255.py:70: Done.
